# Host multiple inference workloads on a SageMaker endpoint with NVIDIA MIG

This notebook demonstrates how to partition the GPUs behind a SageMaker real-time endpoint with NVIDIA Multi-Instance GPU (MIG), place inference components on specific partition types, and scale the components within the endpoint's fixed partition capacity.

You will:

- Configure a mixed MIG layout on an `ml.p4d.24xlarge` endpoint.
- Deploy the same Hugging Face model to three inference components using different MIG profiles.
- Route requests to each inference component independently.
- Compare latency and throughput across the service configurations.
- Run an isolation experiment while another partition is under load.
- Scale an inference component up to the available partition capacity.
- Inspect actionable failure information when SageMaker cannot satisfy a request.

> **Cost warning:** This notebook creates an `ml.p4d.24xlarge` real-time endpoint. Run the cleanup section when you finish.

## How MIG helps inference workloads

MIG divides a physical NVIDIA GPU into hardware-isolated instances. Each partition receives dedicated GPU memory, cache, and compute resources. This is useful for consolidating smaller inference workloads while retaining stronger resource isolation than software-only sharing.

| Mechanism | Partitioning | Memory isolation | Compute isolation | Typical use |
|---|---|---|---|---|
| CUDA streams | Within one process | No | No | Overlap work in one application |
| MPS | Logical sharing | Yes | Percentage-based | Cooperative CUDA processes |
| MIG | Hardware partitioning | Yes | Yes | Predictable multi-workload inference |

MIG profile names encode compute slices and memory, such as `mig-2g.10gb`. Profile names depend on the GPU model and memory size, so a profile valid for an A100-40GB GPU is not necessarily the correct profile for an A100-80GB GPU.

## Architecture and capacity

The endpoint in this notebook uses eight A100-40GB GPUs. Each physical GPU receives the same mixed layout:

```text
ml.p4d.24xlarge: 8 × A100-40GB

Each GPU:
[ mig-4g.20gb ][ mig-2g.10gb ][ mig-1g.10gb ]

Endpoint-wide partition capacity:
8 × mig-4g.20gb
8 × mig-2g.10gb
8 × mig-1g.10gb
```

The endpoint configuration defines the partition layout. Each inference component then requests a partition type and a copy count. Scaling can consume free matching partitions, but it cannot change the partition layout.

## Prerequisites

Before running the notebook, confirm that:

- Your AWS Region and account have access to the accelerator-partitioning feature.
- Your account has sufficient `ml.p4d.24xlarge` endpoint quota and capacity.
- The execution role can create and manage SageMaker models, endpoint configurations, endpoints, and inference components.
- The execution role can pull the selected inference image and download the Hugging Face model.
- If you run outside SageMaker Studio, you provide an execution-role ARN below.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3 pandas

In [ ]:
import concurrent.futures
import json
import math
import re
import statistics
import time
from collections import Counter

import boto3
import pandas as pd
from botocore.exceptions import ClientError
from IPython.display import Markdown, display

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")
sm_runtime = boto3.client("sagemaker-runtime")

print(f"AWS Region: {region}")

## Resource waiters

The helpers below do more than wait for a status transition:

- They impose a timeout instead of polling forever.
- They raise immediately when SageMaker reports a failed state.
- The raised exception includes the service-provided `FailureReason`.
- For inference components, they also detect partial placement where the current copy count is lower than the desired copy count.

In [2]:
IN_PROGRESS_STATUSES = {
    "Creating",
    "Updating",
    "SystemUpdating",
    "RollingBack",
}
FAILURE_STATUSES = {
    "Failed",
    "OutOfService",
    "UpdateRollbackFailed",
    "RollbackFailed",
}


def get_sagemaker_role():
    """Return the IAM role behind the current SageMaker/STS session."""
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    role_arn = re.sub(
        r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$",
        r"\1iam::\2:role/\3",
        arn,
    )
    if role_arn == arn:
        raise RuntimeError(
            "Could not infer a SageMaker execution role from the current STS identity. "
            "Set the role variable explicitly in the next cell."
        )
    return role_arn


def _failure_message(label, name, status, description):
    reason = description.get("FailureReason") or "No FailureReason was returned by SageMaker."
    return f"{label} '{name}' reached status '{status}'. Failure reason: {reason}"


def _wait_for_resource(
    describe_fn,
    name_key,
    status_key,
    label,
    name,
    *,
    success_statuses=("InService",),
    sleep_time=30,
    timeout=3600,
):
    """Wait for a resource and return its final description.

    Failed and unexpected terminal states raise RuntimeError with the
    SageMaker FailureReason. A long-running operation raises TimeoutError
    with the latest observed status and failure information.
    """
    started = time.monotonic()
    last_status = None

    while True:
        description = describe_fn(**{name_key: name})
        status = description[status_key]
        elapsed = time.monotonic() - started

        if status != last_status:
            print(f"{label}: '{name}', status: '{status}', elapsed: {elapsed:.0f}s")
            last_status = status

        if status in success_statuses:
            return description

        if status in FAILURE_STATUSES:
            raise RuntimeError(_failure_message(label, name, status, description))

        if status not in IN_PROGRESS_STATUSES:
            raise RuntimeError(
                _failure_message(label, name, status, description)
                + " The waiter did not recognize this as an in-progress state."
            )

        if elapsed >= timeout:
            raise TimeoutError(
                f"Timed out after {elapsed:.0f}s waiting for {label.lower()} '{name}'. "
                + _failure_message(label, name, status, description)
            )

        time.sleep(sleep_time)


def wait_for_endpoint(endpoint_name, sleep_time=30, timeout=3600):
    return _wait_for_resource(
        sm.describe_endpoint,
        "EndpointName",
        "EndpointStatus",
        "Endpoint",
        endpoint_name,
        sleep_time=sleep_time,
        timeout=timeout,
    )


def wait_for_ic(
    ic_name,
    *,
    expected_copy_count=None,
    sleep_time=30,
    timeout=3600,
):
    """Wait for an inference component and verify its requested capacity.

    expected_copy_count prevents a race where a describe call briefly returns
    the old InService state immediately after a runtime-config update.
    """
    started = time.monotonic()

    while True:
        elapsed = time.monotonic() - started
        remaining_timeout = max(1, timeout - elapsed)
        description = _wait_for_resource(
            sm.describe_inference_component,
            "InferenceComponentName",
            "InferenceComponentStatus",
            "Inference component",
            ic_name,
            sleep_time=sleep_time,
            timeout=remaining_timeout,
        )

        runtime = description.get("RuntimeConfig", {})
        desired = runtime.get("DesiredCopyCount")
        current = runtime.get("CurrentCopyCount")

        if expected_copy_count is not None and desired != expected_copy_count:
            if time.monotonic() - started >= timeout:
                reason = description.get("FailureReason") or "No FailureReason was returned by SageMaker."
                raise TimeoutError(
                    f"Timed out waiting for inference component '{ic_name}' to accept "
                    f"DesiredCopyCount={expected_copy_count}; latest desired count was "
                    f"{desired}. Failure reason: {reason}"
                )
            time.sleep(sleep_time)
            continue

        if desired is not None and current is not None and current != desired:
            reason = description.get("FailureReason") or "No FailureReason was returned by SageMaker."
            raise RuntimeError(
                f"Inference component '{ic_name}' is InService, but SageMaker placed "
                f"{current} of {desired} requested copies. Failure reason: {reason}"
            )

        return description


def _wait_until_not_found(
    describe_fn,
    name_key,
    label,
    name,
    *,
    sleep_time=15,
    timeout=1800,
):
    """Wait until a SageMaker resource can no longer be described."""
    started = time.monotonic()
    while True:
        try:
            description = describe_fn(**{name_key: name})
        except ClientError as error:
            if error.response.get("Error", {}).get("Code") == "ValidationException":
                print(f"{label}: '{name}' deleted")
                return
            raise

        elapsed = time.monotonic() - started
        if elapsed >= timeout:
            status = next(
                (value for key, value in description.items() if key.endswith("Status")),
                "unknown",
            )
            raise TimeoutError(
                f"Timed out waiting for {label.lower()} '{name}' to be deleted. "
                f"Latest status: {status}"
            )
        time.sleep(sleep_time)

## Execution role

In [ ]:
# Set this explicitly when running outside SageMaker Studio.
role = None

if role is None:
    role = get_sagemaker_role()

print(f"SageMaker execution role: {role}")

## Deployment configuration

The mixed partition layout is repeated on every physical GPU. The capacity helper calculates how many partitions of each type are available across the endpoint.

In [4]:
instance = {
    "type": "ml.p4d.24xlarge",
    "num_gpu": 8,
    "gpu": "A100-40GB",
}

partition_layout = [
    "mig-4g.20gb",
    "mig-2g.10gb",
    "mig-1g.10gb",
]

model_id = "Qwen/Qwen3-0.6B"
resource_suffix = time.strftime("%y%m%d-%H%M%S")
model_name = f"mig-model-{resource_suffix}"
endpoint_name = f"mig-endpoint-{resource_suffix}"
endpoint_config_name = f"mig-endpoint-config-{resource_suffix}"
variant_name = "variant-1"
startup_timeout = 600

layout_capacity = {
    profile: count * instance["num_gpu"]
    for profile, count in Counter(partition_layout).items()
}

display(
    pd.DataFrame(
        [
            {
                "MIG profile": profile,
                "Partitions per GPU": partition_layout.count(profile),
                "Physical GPUs": instance["num_gpu"],
                "Endpoint capacity": capacity,
            }
            for profile, capacity in layout_capacity.items()
        ]
    )
)

,MIG profile,Partitions per GPU,Physical GPUs,Endpoint capacity
0,mig-4g.20gb,1,8,8
1,mig-2g.10gb,1,8,8
2,mig-1g.10gb,1,8,8


## vLLM container

The notebook downloads the model directly from Hugging Face when each inference-component copy starts. All three inference components reference the same SageMaker model so that the deployment focuses on partition placement and scaling rather than differences between serving stacks.

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.28.0-gpu-py312-cu130-ubuntu24.04-sagemaker"

environment = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_MAX_MODEL_LEN": "32736",
}

print(f"Model: {model_id}")
print(f"Container: {inference_image}")

## Create the SageMaker model

In [ ]:
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": environment,
    },
)

print(f"Created model: {model_name}")

## Create the MIG-partitioned endpoint

`AcceleratorPartitionLayouts` defines the profile mix on each physical GPU. The capacity reservation below is intentionally hard-coded for the current test.

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": startup_timeout,
            "VariantInstanceProvisionTimeoutInSeconds": startup_timeout,
            "InstanceType": instance["type"],
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            "AcceleratorPartitionLayouts": [
                {"AcceleratorPartitionTypes": partition_layout}
            ],
        }
    ],
)

print(f"Created endpoint configuration: {endpoint_config_name}")

In [ ]:
sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

endpoint_description = wait_for_endpoint(endpoint_name)
print(f"Endpoint is ready: {endpoint_name}")

In [ ]:
endpoint_description = sm.describe_endpoint(EndpointName=endpoint_name)
print(json.dumps(endpoint_description, indent=2, default=str))

## Define the inference components

The initial allocation mirrors three service tiers:

- One copy on a `4g.20gb` partition.
- Eight copies on `2g.10gb`, consuming all matching partitions.
- Four copies on `1g.10gb`, leaving four matching partitions free.

Each copy requires exactly one MIG device.

In [10]:
components = [
    {
        "label": "large-partition",
        "name": f"ic-4g-{resource_suffix}",
        "partition_type": "mig-4g.20gb",
        "copy_count": 1,
    },
    {
        "label": "medium-partition",
        "name": f"ic-2g-{resource_suffix}",
        "partition_type": "mig-2g.10gb",
        "copy_count": 8,
    },
    {
        "label": "small-partition",
        "name": f"ic-1g-{resource_suffix}",
        "partition_type": "mig-1g.10gb",
        "copy_count": 4,
    },
]


def allocation_table(component_definitions):
    requested = Counter()
    for component in component_definitions:
        requested[component["partition_type"]] += component["copy_count"]

    rows = []
    for profile, available in layout_capacity.items():
        used = requested[profile]
        rows.append(
            {
                "MIG profile": profile,
                "Available": available,
                "Requested": used,
                "Remaining": available - used,
                "Fits": used <= available,
            }
        )
    return pd.DataFrame(rows)


requested_allocation = allocation_table(components)
display(requested_allocation)

if not requested_allocation["Fits"].all():
    raise ValueError("The requested inference-component copies exceed the MIG layout capacity.")

,MIG profile,Available,Requested,Remaining,Fits
0,mig-4g.20gb,8,1,7,True
1,mig-2g.10gb,8,8,0,True
2,mig-1g.10gb,8,4,4,True


## Create and place the inference components

In [ ]:
def create_inference_component(component):
    sm.create_inference_component(
        InferenceComponentName=component["name"],
        EndpointName=endpoint_name,
        VariantName=variant_name,
        Specification={
            "ModelName": model_name,
            "StartupParameters": {
                "ModelDataDownloadTimeoutInSeconds": startup_timeout,
                "ContainerStartupHealthCheckTimeoutInSeconds": startup_timeout,
            },
            "ComputeResourceRequirements": {
                "AcceleratorPartitionType": component["partition_type"],
                "MinMemoryRequiredInMb": 1024,
                "NumberOfAcceleratorDevicesRequired": 1,
            },
        },
        RuntimeConfig={"CopyCount": component["copy_count"]},
    )
    return wait_for_ic(
        component["name"],
        expected_copy_count=component["copy_count"],
    )


for component in components:
    print(
        f"Creating {component['name']}: "
        f"{component['copy_count']} × {component['partition_type']}"
    )
    create_inference_component(component)

print("All inference components are ready.")

## Inspect component placement

This summary exposes the desired and current copy counts and preserves any service-provided failure information.

In [12]:
def inference_component_status_table(component_definitions=components):
    rows = []
    for component in component_definitions:
        description = sm.describe_inference_component(
            InferenceComponentName=component["name"]
        )
        runtime = description.get("RuntimeConfig", {})
        requirements = description.get("Specification", {}).get(
            "ComputeResourceRequirements", {}
        )
        rows.append(
            {
                "Component": component["name"],
                "Status": description.get("InferenceComponentStatus"),
                "MIG profile": requirements.get("AcceleratorPartitionType"),
                "Desired copies": runtime.get("DesiredCopyCount"),
                "Current copies": runtime.get("CurrentCopyCount"),
                "Failure reason": description.get("FailureReason"),
            }
        )
    return pd.DataFrame(rows)


display(inference_component_status_table())

,Component,Status,MIG profile,Desired copies,Current copies,Failure reason
0,ic-4g-260917-200037,InService,mig-4g.20gb,1,1,None
1,ic-2g-260917-200037,InService,mig-2g.10gb,8,8,None
2,ic-1g-260917-200037,InService,mig-1g.10gb,4,4,None


## Invoke each inference component

All requests use the same endpoint. `InferenceComponentName` selects the service configuration that receives the request.

In [13]:
def invoke_endpoint_raw(payload, inference_component):
    started = time.perf_counter()
    response = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        InferenceComponentName=inference_component,
        Body=json.dumps(payload),
        ContentType="application/json",
    )
    body = json.loads(response["Body"].read().decode("utf-8"))
    return body, time.perf_counter() - started


def invoke_endpoint(payload, inference_component):
    response, elapsed = invoke_endpoint_raw(payload, inference_component)
    print(f"Response time: {elapsed:.2f}s")

    message = response.get("choices", [{}])[0].get("message", {})
    reasoning = message.get("reasoning_content") or message.get("reasoning")
    content = message.get("content")

    rendered = []
    if reasoning:
        rendered.append(f"### Reasoning\n\n{reasoning}")
    if content:
        rendered.append(f"### Content\n\n{content}")

    if rendered:
        display(Markdown("\n\n---\n\n".join(rendered)))
    else:
        print(json.dumps(response, indent=2))

    return response


payload = {
    "messages": [
        {
            "role": "user",
            "content": "In two sentences, explain why GPU partitioning is useful for inference.",
        }
    ],
    "max_tokens": 96,
    "temperature": 0,
}

In [14]:
for component in components:
    display(
        Markdown(
            f"## {component['label']}: `{component['partition_type']}` "
            f"({component['copy_count']} copies)"
        )
    )
    invoke_endpoint(payload, component["name"])

## large-partition: `mig-4g.20gb` (1 copies)

Response time: 0.46s


### Content

<think>
Okay, the user is asking for two sentences to explain why GPU partitioning is useful for inference. Let me start by recalling what GPU partitioning does. I know that GPUs are powerful, so partitioning might mean dividing the GPU into multiple cores or processing units. But wait, I think it's more about how GPUs are used in inference tasks. Inference usually refers to processing data quickly, so maybe partitioning helps in parallelizing tasks.

First sentence: GPU partition

## medium-partition: `mig-2g.10gb` (8 copies)

Response time: 0.53s


### Content

<think>
Okay, the user is asking for two sentences to explain why GPU partitioning is useful for inference. Let me start by recalling what GPU partitioning does. I know that GPUs are designed for parallel processing, so partitioning might divide the GPU into multiple cores or regions to handle different tasks. 

First sentence: Maybe mention that it allows different tasks to be processed in parallel, which is useful for inference. Second sentence: Highlight how this improves performance by distributing the workload across

## small-partition: `mig-1g.10gb` (4 copies)

Response time: 0.55s


### Content

<think>
Okay, the user is asking for two sentences to explain why GPU partitioning is useful for inference. Let me start by recalling what GPU partitioning does. I know that GPUs are designed for parallel processing, so partitioning allows different parts of the GPU to handle different tasks. For inference, maybe it's about splitting the workload to optimize performance.

First sentence: GPU partitioning enables efficient resource allocation by dividing the GPU into multiple cores or blocks, allowing each part to focus

## Benchmark the service configurations

The benchmark uses the same prompt for every inference component and reports request latency and aggregate request throughput. Because the components have different copy counts, these results describe the complete service configurations—not only the speed of an individual MIG partition.

For meaningful measurements, run several iterations, use representative prompts and output lengths, and treat the first request as warm-up.

In [19]:
BENCHMARK_PAYLOAD = {
    "messages": [
        {
            "role": "user",
            "content": "List three practical benefits of hardware-isolated GPU partitions.",
        }
    ],
    "max_tokens": 12,
    "temperature": 0,
}


def percentile(values, percentile_value):
    if not values:
        return None
    ordered = sorted(values)
    index = max(0, math.ceil(percentile_value / 100 * len(ordered)) - 1)
    return ordered[index]


def benchmark_component(
    inference_component,
    *,
    request_count=48,
    concurrency=4,
    payload=BENCHMARK_PAYLOAD,
):
    latencies = []
    errors = []
    wall_start = time.perf_counter()

    def invoke_once(_):
        _, elapsed = invoke_endpoint_raw(payload, inference_component)
        return elapsed

    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(invoke_once, request_id) for request_id in range(request_count)]
        for future in concurrent.futures.as_completed(futures):
            try:
                latencies.append(future.result())
            except Exception as error:
                errors.append(f"{type(error).__name__}: {error}")

    wall_time = time.perf_counter() - wall_start
    return {
        "Component": inference_component,
        "Requests": request_count,
        "Concurrency": concurrency,
        "Succeeded": len(latencies),
        "Failed": len(errors),
        "p50 latency (s)": statistics.median(latencies) if latencies else None,
        "p95 latency (s)": percentile(latencies, 95),
        "Requests/s": len(latencies) / wall_time if wall_time else None,
        "First error": errors[0] if errors else None,
    }


RUN_BENCHMARKS = True

In [20]:
benchmark_results = []

if RUN_BENCHMARKS:
    for component in components:
        # Warm up the selected inference component before recording results.
        invoke_endpoint_raw(BENCHMARK_PAYLOAD, component["name"])
        result = benchmark_component(
            component["name"],
            request_count=64,
            concurrency=min(4, component["copy_count"]),
        )
        result["MIG profile"] = component["partition_type"]
        result["Configured copies"] = component["copy_count"]
        benchmark_results.append(result)

    display(pd.DataFrame(benchmark_results))
else:
    print("Benchmarks are disabled. Set RUN_BENCHMARKS=True to run them.")

,Component,Requests,Concurrency,Succeeded,Failed,p50 latency (s),p95 latency (s),Requests/s,First error,MIG profile,Configured copies
0,ic-4g-260917-200037,64,1,64,0,0.054385,0.056466,18.311480,None,mig-4g.20gb,1
1,ic-2g-260917-200037,64,4,64,0,0.076949,0.082630,51.184299,None,mig-2g.10gb,8
2,ic-1g-260917-200037,64,4,64,0,0.087480,0.095478,45.375419,None,mig-1g.10gb,4


## Isolation experiment

This experiment first measures the `4g.20gb` component by itself. It then generates concurrent load against all eight `2g.10gb` copies while measuring the `4g.20gb` component again.

MIG isolates GPU compute and memory resources, but the workloads still share host CPU, networking, and other endpoint resources. The result should therefore be interpreted as an end-to-end service measurement, not a promise of perfectly unchanged latency.

In [21]:
def run_isolation_experiment(observed_component, noisy_component):
    baseline = benchmark_component(
        observed_component,
        request_count=8,
        concurrency=2,
    )
    baseline["Scenario"] = "Observed component only"

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        noisy_future = executor.submit(
            benchmark_component,
            noisy_component,
            request_count=32,
            concurrency=8,
        )
        time.sleep(1)
        under_load = benchmark_component(
            observed_component,
            request_count=8,
            concurrency=2,
        )
        noisy_result = noisy_future.result()

    under_load["Scenario"] = "While another MIG profile is under load"
    noisy_result["Scenario"] = "Background load generator"
    return pd.DataFrame([baseline, under_load, noisy_result])


RUN_ISOLATION_TEST = RUN_BENCHMARKS

if RUN_ISOLATION_TEST:
    isolation_results = run_isolation_experiment(
        observed_component=components[0]["name"],
        noisy_component=components[1]["name"],
    )
    display(isolation_results)
else:
    print("Isolation test is disabled.")

,Component,Requests,Concurrency,Succeeded,Failed,p50 latency (s),p95 latency (s),Requests/s,First error,Scenario
0,ic-4g-260917-200037,8,2,8,0,0.061219,0.071096,31.379072,None,Observed component only
1,ic-4g-260917-200037,8,2,8,0,0.064256,0.074494,30.192167,None,While another MIG profile is under load
2,ic-2g-260917-200037,32,8,32,0,0.080266,0.140112,82.395322,None,Background load generator


## Scale within the fixed MIG layout

The endpoint contains eight `mig-4g.20gb` partitions and initially uses one. The following cell scales that inference component to eight copies, consuming every matching partition. The other profile pools are unaffected.

In [22]:
large_component = components[0]
target_copy_count = layout_capacity[large_component["partition_type"]]

scale_results = []
if RUN_BENCHMARKS:
    before_scale = benchmark_component(
        large_component["name"],
        request_count=16,
        concurrency=8,
    )
    before_scale["Configuration"] = f"Before scaling ({large_component['copy_count']} copy)"
    scale_results.append(before_scale)

sm.update_inference_component_runtime_config(
    InferenceComponentName=large_component["name"],
    DesiredRuntimeConfig={"CopyCount": target_copy_count},
)
wait_for_ic(
    large_component["name"],
    expected_copy_count=target_copy_count,
)
large_component["copy_count"] = target_copy_count

print(
    f"Scaled {large_component['name']} to {target_copy_count} "
    f"copies on {large_component['partition_type']}."
)
display(allocation_table(components))
display(inference_component_status_table())

if RUN_BENCHMARKS:
    after_scale = benchmark_component(
        large_component["name"],
        request_count=16,
        concurrency=8,
    )
    after_scale["Configuration"] = f"After scaling ({target_copy_count} copies)"
    scale_results.append(after_scale)
    display(pd.DataFrame(scale_results))

Inference component: 'ic-4g-260917-200037', status: 'Updating', elapsed: 0s
Inference component: 'ic-4g-260917-200037', status: 'InService', elapsed: 332s
Scaled ic-4g-260917-200037 to 8 copies on mig-4g.20gb.


,MIG profile,Available,Requested,Remaining,Fits
0,mig-4g.20gb,8,8,0,True
1,mig-2g.10gb,8,8,0,True
2,mig-1g.10gb,8,4,4,True


,Component,Status,MIG profile,Desired copies,Current copies,Failure reason
0,ic-4g-260917-200037,InService,mig-4g.20gb,8,8,None
1,ic-2g-260917-200037,InService,mig-2g.10gb,8,8,None
2,ic-1g-260917-200037,InService,mig-1g.10gb,4,4,None


,Component,Requests,Concurrency,Succeeded,Failed,p50 latency (s),p95 latency (s),Requests/s,First error,Configuration
0,ic-4g-260917-200037,16,8,16,0,0.075360,0.120951,77.311753,None,Before scaling (1 copy)
1,ic-4g-260917-200037,16,8,16,0,0.105684,0.168002,66.566003,None,After scaling (8 copies)


## Optional: demonstrate capacity exhaustion

The waiter detects both a failed inference component and partial placement where `CurrentCopyCount` is lower than `DesiredCopyCount`. Set the flag below to request one more `4g.20gb` copy than the endpoint can host. The expected exception includes SageMaker's `FailureReason`, after which the cell restores the valid copy count.

In [23]:
RUN_OVERCOMMIT_DEMO = True

if RUN_OVERCOMMIT_DEMO:
    invalid_copy_count = target_copy_count + 1
    try:
        sm.update_inference_component_runtime_config(
            InferenceComponentName=large_component["name"],
            DesiredRuntimeConfig={"CopyCount": invalid_copy_count},
        )
        wait_for_ic(
            large_component["name"],
            expected_copy_count=invalid_copy_count,
        )
    except RuntimeError as error:
        display(Markdown(f"### Expected capacity error\n\n```text\n{error}\n```"))
    finally:
        sm.update_inference_component_runtime_config(
            InferenceComponentName=large_component["name"],
            DesiredRuntimeConfig={"CopyCount": target_copy_count},
        )
        wait_for_ic(
            large_component["name"],
            expected_copy_count=target_copy_count,
        )
        print(f"Restored {large_component['name']} to {target_copy_count} copies.")
else:
    print("Overcommit demonstration skipped. Set RUN_OVERCOMMIT_DEMO=True to run it.")

Inference component: 'ic-4g-260917-200037', status: 'Updating', elapsed: 0s
Inference component: 'ic-4g-260917-200037', status: 'InService', elapsed: 30s


### Expected capacity error

```text
Inference component 'ic-4g-260917-200037' is InService, but SageMaker placed 8 of 9 requested copies. Failure reason: With the currently available resources, SageMaker was able to deploy 8 out of 9 requested copies of inference components. Please update resource requirements, remove existing inference components, or increase the number of instances for this endpoint.
```

Inference component: 'ic-4g-260917-200037', status: 'Updating', elapsed: 0s
Inference component: 'ic-4g-260917-200037', status: 'InService', elapsed: 30s
Restored ic-4g-260917-200037 to 8 copies.


## Final resource state

Use this summary together with the SageMaker endpoint logs and metrics when investigating placement, startup, or runtime behavior.

In [ ]:
display(inference_component_status_table())

endpoint_description = sm.describe_endpoint(EndpointName=endpoint_name)
print(
    json.dumps(
        {
            "EndpointName": endpoint_description.get("EndpointName"),
            "EndpointStatus": endpoint_description.get("EndpointStatus"),
            "EndpointConfigName": endpoint_description.get("EndpointConfigName"),
            "ProductionVariants": endpoint_description.get("ProductionVariants"),
            "FailureReason": endpoint_description.get("FailureReason"),
        },
        indent=2,
        default=str,
    )
)

## Cleanup

The cleanup helper tolerates resources that were never created or were already removed. It waits for inference-component and endpoint deletion before deleting dependent resources.

In [25]:
def is_not_found(error):
    return (
        isinstance(error, ClientError)
        and error.response.get("Error", {}).get("Code") == "ValidationException"
    )


def cleanup_resources():
    for component in components:
        try:
            sm.delete_inference_component(
                InferenceComponentName=component["name"]
            )
            print(f"Deleting inference component: {component['name']}")
        except ClientError as error:
            if not is_not_found(error):
                raise

    for component in components:
        _wait_until_not_found(
            sm.describe_inference_component,
            "InferenceComponentName",
            "Inference component",
            component["name"],
        )

    try:
        sm.delete_endpoint(EndpointName=endpoint_name)
        print(f"Deleting endpoint: {endpoint_name}")
    except ClientError as error:
        if not is_not_found(error):
            raise

    _wait_until_not_found(
        sm.describe_endpoint,
        "EndpointName",
        "Endpoint",
        endpoint_name,
    )

    try:
        sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
        print(f"Deleted endpoint configuration: {endpoint_config_name}")
    except ClientError as error:
        if not is_not_found(error):
            raise

    try:
        sm.delete_model(ModelName=model_name)
        print(f"Deleted model: {model_name}")
    except ClientError as error:
        if not is_not_found(error):
            raise

    print("Cleanup complete.")

In [ ]:
# Run this cell when you are finished with the endpoint.
cleanup_resources()